# Discesa stocastica del gradiente



## Preparazione in Colab

1. Salva una copia del notebook nel tuo Drive.
2. Esegui la cella seguente e attendi «Preparazione completata».
3. Prosegui dall’alto verso il basso e completa gli esercizi.

La cella carica i dataset e i controlli dal repository del corso. Non devi installare Python né caricare manualmente i CSV. Se la sessione viene riavviata, esegui nuovamente la preparazione.

In [ ]:
from pathlib import Path
import hashlib
import importlib.util
from urllib.request import urlopen
from IPython.display import display, Markdown

course_source = 'https://raw.githubusercontent.com/prof-davidaffo/kaggle-learntools/master'
course_root = (Path("/content") if Path("/content").is_dir() else Path.home()) / "ai-lab-colab"
course_root.mkdir(parents=True, exist_ok=True)
bootstrap_path = course_root / "_bootstrap.py"
bootstrap_hash = '24c516651bf205b1c9511923ad4971fdca39a6248568d2b3aa46c198e64d1821'
if not bootstrap_path.is_file() or hashlib.sha256(bootstrap_path.read_bytes()).hexdigest() != bootstrap_hash:
    with urlopen(course_source + "/ai_lab/tools/colab_runtime.py", timeout=90) as response:
        bootstrap_data = response.read()
    if hashlib.sha256(bootstrap_data).hexdigest() != bootstrap_hash:
        raise RuntimeError("Course setup changed. Reopen the latest notebook from the course site.")
    bootstrap_path.write_bytes(bootstrap_data)
bootstrap_spec = importlib.util.spec_from_file_location("ai_lab_colab_bootstrap", bootstrap_path)
bootstrap = importlib.util.module_from_spec(bootstrap_spec)
bootstrap_spec.loader.exec_module(bootstrap)
bootstrap.download_runtime(course_source + "/ai_lab/colab/runtime.json", course_source, course_root)
bootstrap.setup(course_root, globals(), deep_learning=True)
display(Markdown("**Preparazione completata.** Puoi eseguire le celle della lezione."))


# Introduzione
Hai imparato a costruire reti dense. All'inizio i pesi sono casuali: la rete non ha ancora appreso relazioni. **Addestrarla** significa modificarli per trasformare le feature nei target attesi.

Nel dataset dei cereali, per esempio, vogliamo prevedere le calorie da zuccheri, fibre e proteine. Dopo un buon addestramento i pesi rappresentano una relazione ricavata dai dati. Oltre agli esempi di training servono:

* Una **loss function**, che misura l'errore.
* Un **optimizer**, che decide come aggiornare i pesi.

# La funzione di perdita
L'architettura stabilisce come calcolare l'uscita; la **loss** indica quale problema risolvere, confrontando previsione e target reale.

Per problemi di **regressione**, come calorie, qualità del vino, prezzo di una casa o consumo di un'auto, una scelta comune è il **MAE**: per ogni esempio calcoliamo `abs(y_true - y_pred)` e poi facciamo la media.

![Distanze fra punti e previsione](https://storage.googleapis.com/kaggle-media/learn/images/VDcvkZN.png)

Altre loss per la regressione sono **MSE**, errore quadratico medio, e **Huber**. Durante l'addestramento cerchiamo pesi che riducano la loss: essa definisce l'obiettivo della rete.

# L'ottimizzatore: discesa stocastica del gradiente
L'**optimizer** è l'algoritmo che aggiorna i pesi per minimizzare la loss. Molti algoritmi di deep learning appartengono alla famiglia della **stochastic gradient descent**, SGD. Ogni passo:

1. Seleziona un gruppo di esempi e calcola le previsioni.
2. Misura la loss rispetto ai target.
3. Aggiorna i pesi in una direzione che tende a ridurre la loss.

Ripetiamo finché il risultato è soddisfacente o non migliora più.

![Addestramento batch dopo batch](https://storage.googleapis.com/kaggle-media/learn/images/rFI1tIk.gif)

Il gruppo di esempi è un **minibatch**, spesso chiamato semplicemente *batch*. Un passaggio completo sui dati è un'**epoca**, o *epoch*. Nell'animazione i punti chiari rappresentano tutti i dati e quelli scuri il batch corrente. SGD aggiorna la pendenza `w` e l'intercetta `b`; progressivamente la retta si adatta e la loss diminuisce.

## Learning rate e batch size
Il **learning rate** controlla l'ampiezza degli aggiornamenti. Un valore piccolo richiede in genere più passi per convergere. Il learning rate e il numero di esempi per batch influenzano tempi, oscillazioni e risultato del training; la loro interazione va esplorata.

**Adam** è un ottimizzatore della famiglia SGD con adattamento del passo, spesso una buona scelta iniziale senza una ricerca estesa dei parametri.

## Aggiungere loss e optimizer
Dopo la definizione del modello usiamo `compile`:

```python
model.compile(optimizer="adam", loss="mae")
```

Le stringhe identificano implementazioni Keras. Per regolare i parametri puoi anche creare gli oggetti dell'API; qui bastano i valori predefiniti.

Il **gradiente** è il vettore delle derivate e indica la direzione di aumento più rapido della loss. La **discesa** procede in senso opposto verso un minimo. **Stocastica** richiama la casualità nel campionamento dei minibatch.


# Esempio: qualità del vino rosso
Il dataset **Red Wine Quality** contiene misure fisico-chimiche di circa 1.600 vini portoghesi e valutazioni ottenute con degustazioni alla cieca. Quanto possiamo prevedere la qualità percepita dalle misure?

La cella seguente prepara i dati. Nell'esempio originale le feature vengono riportate nell'intervallo **\([0,1]\)**: le reti tendono a funzionare meglio con ingressi su scale confrontabili. Approfondiremo nella lezione 5.


In [ ]:

import pandas as pd
from IPython.display import display

red_wine = pd.read_csv('../input/dl-course-data/red-wine.csv')

# Suddividi i dati in training e validazione.
df_train = red_wine.sample(frac=0.7, random_state=0)
df_valid = red_wine.drop(df_train.index)
display(df_train.head(4))

# Riporta i valori nell'intervallo [0, 1].
max_ = df_train.max(axis=0)
min_ = df_train.min(axis=0)
df_train = (df_train - min_) / (max_ - min_)
df_valid = (df_valid - min_) / (max_ - min_)

# Separa feature e target.
X_train = df_train.drop('quality', axis=1)
X_valid = df_valid.drop('quality', axis=1)
y_train = df_train['quality']
y_valid = df_valid['quality']

Quanti ingressi servono? Conta le colonne della matrice delle feature, **escludendo il target `quality`**.


In [ ]:
print(X_train.shape)

Undici colonne significano undici ingressi. La rete proposta ha tre strati nascosti e più di 1.500 neuroni: può apprendere relazioni articolate.


In [ ]:
from tensorflow import keras
from tensorflow.keras import layers

model = keras.Sequential([
    layers.Dense(512, activation='relu', input_shape=[11]),
    layers.Dense(512, activation='relu'),
    layers.Dense(512, activation='relu'),
    layers.Dense(1),
])

L'architettura va scelta con un processo di confronto: parti semplice e usa la **validation loss** come guida. Dopo aver definito il modello aggiungiamo optimizer e loss con `compile`.


In [ ]:
model.compile(
    optimizer='adam',
    loss='mae',
)

Il training usa **256 righe per batch** e **10 epoche**, cioè dieci passaggi completi sul dataset.


In [ ]:
history = model.fit(
    X_train, y_train,
    validation_data=(X_valid, y_valid),
    batch_size=256,
    epochs=10,
)

Keras mostra la loss durante l'esecuzione. Per interpretarla è utile un grafico. `fit` restituisce un oggetto **History**, che conserva le misure: convertiamole in un DataFrame pandas per disegnarle.


In [ ]:
import pandas as pd

# Converti lo storico del training in un DataFrame.
history_df = pd.DataFrame(history.history)
# Usa il metodo plot di pandas.
history_df['loss'].plot();

La loss tende ad appiattirsi. Una curva quasi orizzontale indica che, con la configurazione corrente, ulteriori epoche potrebbero offrire pochi benefici: valuta anche il comportamento sui dati di validazione.


Ora **[addestra una rete con SGD](https://prof-davidaffo.github.io/kaggle-learntools/corsi/deep_learning_intro/ex3.html)**.
